# Four stocks: preprocessing and SDE-GAN

Prepare data, train the model, or inspect saved results. Choose `MODE` in setup
and run from the top using the `sandbox/.venv` kernel. The default `"inspect"`
loads saved models. See [the spec](sde_gan_spec.md) for the full method.

Generated paths describe 20-session scenarios from the training-period mixture
of market conditions. Keep the test split for the final evaluation.

In [ ]:
from pathlib import Path
from itertools import combinations
import json
import os
import sys
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = next(
    directory for directory in [Path.cwd(), *Path.cwd().parents]
    if (directory / "sandbox" / "4stocks_experiment" / "sde_gan_spec.md").is_file()
)
experiment_dir = root / "sandbox" / "4stocks_experiment"
data_dir = experiment_dir / "data"
baseline_results_dir = experiment_dir / "results" / "sde_gan"
TEMPORAL_PENALTY_WEIGHT = 30.0  # Controlled trial; the previous experiment used 10.
results_dir = baseline_results_dir / f"temporal_penalty_{TEMPORAL_PENALTY_WEIGHT:g}"
INSPECT_CHECKPOINT_STEP = None  # None: MMD-selected model; integer: inspect that saved step.
STOCKS = ["KONGSBERG GRUPPEN", "FRONTLINE", "WILH Wilhelmsen B", "TOMRA SYSTEMS"]
SPLITS = ["train", "validation", "test"]
WINDOW_LENGTH = 20
STRIDE = 1
OBSERVATION_TIMES = np.arange(WINDOW_LENGTH)
TRAINING_SEEDS = [0]  # Test the stronger penalty on seed 0 before repeating other seeds.
SAMPLE_SEED = 100
N_PATHS = 1000
WEIGHTS = np.full(4, 0.25)

# prepare: data only; inspect: load saved fits; smoke: short synthetic fit;
# train: smoke + full stock fits; test: explicit final held-out evaluation.
MODE = "inspect"
if MODE not in {"prepare", "inspect", "smoke", "train", "test"}:
    raise ValueError("MODE must be prepare, inspect, smoke, train, or test")

print(f"Python: {sys.executable}")
print(f"Experiment: {experiment_dir}")

def inspection_run_dir(seed: int) -> Path:
    """Choose a saved validation step for downstream inspection without refitting."""
    directory = results_dir / f"seed_{seed}"
    if INSPECT_CHECKPOINT_STEP is not None:
        directory = directory / "checkpoints" / f"step_{INSPECT_CHECKPOINT_STEP:04d}"
    return directory


## Load and check returns

Load the three CSV splits and check their dates and stock columns. Keep missing
rows so later steps cannot join returns across gaps. Returns are fractions:
`0.01` means 1%.

In [ ]:
def validate_frame(frame: pd.DataFrame, name: str) -> None:
    """Check the CSV schema and chronological row order without changing rows."""
    if len(frame.columns) != 4 or set(frame.columns) != set(STOCKS):
        raise ValueError(f"{name}: expected exactly the four stock columns")
    if frame.empty or frame.index.hasnans:
        raise ValueError(f"{name}: missing dates or empty split")
    if not frame.index.is_unique or not frame.index.is_monotonic_increasing:
        raise ValueError(f"{name}: dates must be unique and increasing")


def to_log_returns(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    """Transform simple returns while retaining zeros and missing entries."""
    invalid = frame.le(-1)
    if invalid.any().any():
        rows, cols = np.where(invalid.to_numpy())
        details = [(frame.index[i].strftime("%Y-%m-%d"), frame.columns[j])
                   for i, j in zip(rows[:10], cols[:10])]
        raise ValueError(f"{name}: simple return <= -1 at {details}")
    return np.log1p(frame)


def fit_scaler(training_logs: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """Fit each stock on its nonmissing training rows, before windowing."""
    mean = training_logs.mean()
    std = training_logs.std(ddof=1)
    if not np.isfinite(mean).all() or not (np.isfinite(std) & std.gt(0)).all():
        raise ValueError("Training means must be finite and sample stds positive and finite")
    return mean, std


returns = {}
for split in SPLITS:
    frame = pd.read_csv(data_dir / f"{split}_returns.csv")
    frame["Date"] = pd.to_datetime(frame["Date"], format="%Y-%m-%d", errors="raise")
    frame = frame.set_index("Date")
    validate_frame(frame, split)
    returns[split] = frame.loc[:, STOCKS].astype(np.float64)
for left, right in zip(SPLITS, SPLITS[1:]):
    if returns[left].index[-1] >= returns[right].index[0]:
        raise ValueError(f"Overlapping or unordered {left}/{right} split dates")

split_audit = pd.DataFrame([
    {"split": split, "rows": len(frame), "first_date": frame.index[0],
     "last_date": frame.index[-1], "missing_entries": int(frame.isna().sum().sum())}
    for split, frame in returns.items()
]).set_index("split")
display(split_audit)
display(pd.concat({s: pd.DataFrame({"missing": f.isna().sum(), "zeros": f.eq(0).sum()})
                   for s, f in returns.items()}, names=["split", "stock"]))

## Transform and scale returns

Convert returns to log returns, then scale each stock using its training mean
and sample standard deviation. Reuse those training values for every split to
avoid using held-out data to fit the scaler.

In [ ]:
log_returns = {s: to_log_returns(f, s) for s, f in returns.items()}
training_mean, training_std = fit_scaler(log_returns["train"])
standardized = {s: (f - training_mean) / training_std for s, f in log_returns.items()}
scaling_table = pd.DataFrame({
    "training_count": log_returns["train"].count(),
    "mean_log_return": training_mean,
    "sample_std_log_return": training_std,
})
display(scaling_table)
np.testing.assert_allclose(standardized["train"].mean(), 0, atol=1e-12)
np.testing.assert_allclose(standardized["train"].std(ddof=1), 1, atol=1e-12)
for split in SPLITS:
    assert log_returns[split].isna().equals(returns[split].isna())
    assert log_returns[split].eq(0).equals(returns[split].eq(0))

## Build 20-session windows

Keep only windows with 20 consecutive sessions and complete returns for all
four stocks. Build each split separately so windows cannot cross gaps or split
boundaries. Each array has shape `(windows, 20, 4)`.

In [ ]:
def extract_windows(frame: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    """Keep complete consecutive windows and their matching calendar bounds."""
    values = frame.to_numpy(dtype=np.float64)
    paths, dates = [], []
    for start in range(0, len(frame) - WINDOW_LENGTH + 1, STRIDE):
        stop = start + WINDOW_LENGTH
        candidate = values[start:stop]
        if np.isfinite(candidate).all():
            paths.append(candidate)
            dates.append((frame.index[start], frame.index[stop - 1]))
    if not paths:
        raise ValueError("No complete 20-session windows in this split")
    with np.errstate(over="ignore"):
        array = np.ascontiguousarray(np.stack(paths), dtype=np.float32)
    if not np.isfinite(array).all():
        raise ValueError("Standardized windows exceed finite float32 range")
    metadata = pd.DataFrame(dates, columns=["start_date", "end_date"])
    metadata.index.name = "window_index"
    return array, metadata


windows, window_dates, counts = {}, {}, []
for split in SPLITS:
    try:
        windows[split], window_dates[split] = extract_windows(standardized[split])
    except ValueError as error:
        raise ValueError(f"{split}: {error}") from error
    candidates = max(0, (len(returns[split]) - WINDOW_LENGTH) // STRIDE + 1)
    counts.append({"split": split, "candidates": candidates, "retained": len(windows[split]),
                   "rejected": candidates - len(windows[split])})
    assert windows[split].shape[1:] == (20, 4)
    assert windows[split].dtype == np.float32 and windows[split].flags.c_contiguous
    assert window_dates[split]["start_date"].ge(returns[split].index[0]).all()
    assert window_dates[split]["end_date"].le(returns[split].index[-1]).all()
train_windows = windows["train"]
validation_windows = windows["validation"]
test_windows = windows["test"]
window_counts = pd.DataFrame(counts).set_index("split")
display(window_counts)
print({s: a.shape for s, a in windows.items()})

## Check preprocessing

Use small examples to check missing values, invalid returns, window boundaries,
and scaling. Reconstruct observed returns to catch mistakes before training.

In [ ]:
def expect_value_error(operation: Callable[[], object]) -> None:
    """Fail a fixture check if an invalid input is accepted."""
    try:
        operation()
    except ValueError:
        return
    raise AssertionError("Expected ValueError")


fixture = pd.DataFrame(
    np.arange(45 * 4, dtype=float).reshape(45, 4) / 1000,
    index=pd.date_range("2000-01-01", periods=45), columns=STOCKS,
)
fixture.iloc[22, 1] = np.nan
fixture_windows, fixture_dates = extract_windows(fixture)
assert len(fixture_windows) == 6  # starts 0, 1, 2 and 23, 24, 25
assert not ((fixture_dates.start_date <= fixture.index[22]) &
            (fixture_dates.end_date >= fixture.index[22])).any()
np.testing.assert_array_equal(fixture_windows[3], fixture.iloc[23:43].to_numpy(np.float32))
assert len(extract_windows(fixture.iloc[:20])[0]) == 1
assert len(extract_windows(fixture.iloc[23:43])[0]) == 1
expect_value_error(lambda: extract_windows(fixture.iloc[:19]))
expect_value_error(lambda: extract_windows(fixture * np.nan))
expect_value_error(lambda: extract_windows(pd.DataFrame(np.inf, index=fixture.index, columns=STOCKS)))
expect_value_error(lambda: to_log_returns(fixture.fillna(-1), "fixture"))
expect_value_error(lambda: to_log_returns(fixture.fillna(-1.1), "fixture"))
expect_value_error(lambda: fit_scaler(fixture.fillna(0) * 0))
expect_value_error(lambda: validate_frame(fixture.iloc[::-1], "fixture"))
expect_value_error(lambda: validate_frame(pd.concat([fixture, fixture]), "fixture"))
assert to_log_returns(fixture, "fixture").iloc[0, 0] == 0
assert np.isnan(to_log_returns(fixture, "fixture").iloc[22, 1])
fixture_mean, fixture_std = fit_scaler(fixture)
shifted_holdout = (fixture + 10 - fixture_mean) / fixture_std
assert shifted_holdout.mean().min() > 1  # not independently centered
np.testing.assert_array_equal(fixture_mean, fit_scaler(fixture)[0])
for split in SPLITS:
    # Reconstruct source windows directly by their dates, independently of scaling.
    source = np.stack([returns[split].loc[row.start_date:row.end_date].to_numpy()
                       for row in window_dates[split].itertuples()])
    recovered = np.expm1(windows[split].astype(np.float64) * training_std.to_numpy()
                         + training_mean.to_numpy())
    np.testing.assert_allclose(recovered, source, rtol=1e-6, atol=1e-7)
print("Preprocessing fixtures and source-window round trips passed.")

## Save prepared data

Save the arrays, window dates, stock order, and training scaler. Reload them to
check that later runs can recover the same inputs and original return units.

In [ ]:
preprocessing_metadata = {
    "stock_order": STOCKS, "transform": "log1p", "ddof": 1,
    "training_count": scaling_table.training_count.astype(int).tolist(),
    "training_mean": training_mean.tolist(), "training_std": training_std.tolist(),
    "window_length": WINDOW_LENGTH, "stride": STRIDE,
    "observation_times": OBSERVATION_TIMES.tolist(), "time_unit": "market session",
    "calendar": "Observed source calendar corrected in adjusted_price_histories.ipynb",
    "splits": {
        s: {"source": f"{s}_returns.csv", "first_date": f.index[0].strftime("%Y-%m-%d"),
            "last_date": f.index[-1].strftime("%Y-%m-%d"), "rows": len(f),
            **{k: int(v) for k, v in window_counts.loc[s].items()}}
        for s, f in returns.items()
    },
}
all_window_dates = pd.concat(window_dates, names=["split"]).reset_index()
data_dir.mkdir(exist_ok=True)
np.savez_compressed(data_dir / "prepared_windows.npz",
                    **{f"{s}_windows": a for s, a in windows.items()})
all_window_dates.to_csv(data_dir / "window_dates.csv", index=False, date_format="%Y-%m-%d")
(data_dir / "preprocessing.json").write_text(
    json.dumps(preprocessing_metadata, indent=2, allow_nan=False) + "\n"
)
with np.load(data_dir / "prepared_windows.npz", allow_pickle=False) as saved:
    assert set(saved.files) == {f"{s}_windows" for s in SPLITS}
    for split in SPLITS:
        assert saved[f"{split}_windows"].dtype == np.float32
        np.testing.assert_array_equal(saved[f"{split}_windows"], windows[split])
pd.testing.assert_frame_equal(
    pd.read_csv(data_dir / "window_dates.csv", parse_dates=["start_date", "end_date"]),
    all_window_dates,
)
assert json.loads((data_dir / "preprocessing.json").read_text()) == preprocessing_metadata
print(f"Saved and reloaded prepared arrays, dates, and scaler in {data_dir}")

## Inspect training coverage

Show missing observations, retained windows over time, and example paths.
Removing incomplete windows and repeating overlapping dates changes their
weights, so window statistics can differ from the original training-row statistics.

In [ ]:
display(pd.DataFrame({
    "row_mean": standardized["train"].mean(),
    "row_sample_std": standardized["train"].std(ddof=1),
    "window_mean": train_windows.reshape(-1, 4).mean(axis=0),
    "window_sample_std": train_windows.reshape(-1, 4).std(axis=0, ddof=1),
}))
fig, axes = plt.subplots(3, 1, figsize=(13, 9), constrained_layout=True)
axes[0].imshow(returns["train"].isna().to_numpy().T, aspect="auto", interpolation="nearest",
               extent=(0, len(returns["train"]), 3.5, -0.5), cmap="Greys", vmin=0, vmax=1)
axes[0].set(yticks=range(4), yticklabels=STOCKS, xlabel="Training session row",
            title="Missing returns (black); calendar rows are preserved")
coverage = pd.Series(0, index=returns["train"].index)
coverage.loc[pd.DatetimeIndex(window_dates["train"].start_date)] = 1
coverage.resample("YS").sum().plot.bar(ax=axes[1])
axes[1].set(title="Retained training-window starts per year", ylabel="Windows", xlabel="Year")
axes[1].set_xticklabels([str(t.year) for t in coverage.resample("YS").sum().index], rotation=60)
for index in np.unique(np.linspace(0, len(train_windows) - 1, 3, dtype=int)):
    row = window_dates["train"].iloc[index]
    axes[2].plot(OBSERVATION_TIMES, train_windows[index, :, 0],
                 label=f"{row.start_date:%Y-%m-%d} to {row.end_date:%Y-%m-%d}")
axes[2].set(title=f"Example standardized training windows: {STOCKS[0]}",
            xlabel="Observation time (sessions)", ylabel="Standardized log return")
axes[2].legend()
plt.show()
plt.close(fig)

## Set up the model and run a short check

Load the model with the `sandbox/.venv` kernel. Restart the kernel after changing
model code. Numerical tests are in `test_sde_gan.py`.

`MODE = "smoke"` runs 50 updates on synthetic data to check training, sampling,
and saving. `"train"` runs this check before the stock fit. Review runtime and
Activity Monitor's memory pressure before committing to longer experiments.

In [ ]:
if MODE != "prepare":
    os.environ["JAX_PLATFORMS"] = "cpu"
    if str(experiment_dir) not in sys.path:
        sys.path.insert(0, str(experiment_dir))
    try:
        from sde_gan import SDEGAN, SDEGANConfig
        if "retain_validation_checkpoints" not in SDEGANConfig.__dataclass_fields__:
            raise RuntimeError("The kernel has cached old model modules. Restart the kernel and run from setup.")
    except ModuleNotFoundError as error:
        raise RuntimeError(
            f"Model dependencies unavailable in {sys.executable}; select sandbox/.venv "
            "and ensure sde_gan.py, training.py, and architecture.py are present."
        ) from error
else:
    print("Model imports and training disabled; preprocessing is complete.")

In [ ]:
def verify_checkpoint(model: Any, directory: Path) -> None:
    """Check finite sampling, explicit seeds, and inference restoration."""
    first = model.sample(n_paths=17, seed=SAMPLE_SEED)
    assert first.shape == (17, 20, 4) and np.isfinite(first).all()
    np.testing.assert_array_equal(first, model.sample(n_paths=17, seed=SAMPLE_SEED))
    assert not np.array_equal(first, model.sample(n_paths=17, seed=SAMPLE_SEED + 1))
    model.save(directory)
    restored = SDEGAN.load(directory)
    np.testing.assert_array_equal(first, restored.sample(n_paths=17, seed=SAMPLE_SEED))


if MODE in {"smoke", "train"}:
    import threading
    from time import perf_counter
    import psutil

    rng = np.random.default_rng(2026)
    chol = np.linalg.cholesky(0.5 * np.eye(4) + 0.5 * np.ones((4, 4)))
    synthetic = (rng.normal(size=(192, 20, 4)) @ chol.T).astype(np.float32)
    smoke_model = SDEGAN(SDEGANConfig(generator_steps=50, validation_every=50,
                                          temporal_penalty_weight=TEMPORAL_PENALTY_WEIGHT,
                                          retain_validation_checkpoints=True))
    rss_records = []
    stop_monitor = threading.Event()
    process = psutil.Process()
    started = perf_counter()

    def record_rss() -> None:
        """Sample process memory during compilation, fitting, and validation."""
        while True:
            rss_records.append((perf_counter() - started, process.memory_info().rss / 2**20))
            if stop_monitor.wait(1):
                break

    monitor = threading.Thread(target=record_rss, daemon=True)
    monitor.start()
    try:
        smoke_history = pd.DataFrame(smoke_model.fit(synthetic[:128], synthetic[128:], seed=0))
        for record in smoke_model.history:
            assert all(np.isfinite(v) for v in record.values() if v is not None)
        assert set(smoke_history.loc[smoke_history.validation_mmd2.notna(), "step"]) == {0, 50}
        smoke_dir = results_dir / "smoke" / "seed_0"
        verify_checkpoint(smoke_model, smoke_dir)
    finally:
        stop_monitor.set()
        monitor.join()
    rss_frame = pd.DataFrame(rss_records, columns=["elapsed_seconds", "rss_mib"])
    rss_frame.to_csv(smoke_dir / "process_memory.csv", index=False)
    cycle_seconds = float(smoke_history.loc[smoke_history.step >= 2, "cycle_seconds"].median())
    display(pd.Series({"median_cycle_seconds_after_compilation": cycle_seconds,
                       "peak_sampled_rss_mib": rss_frame.rss_mib.max(),
                       "estimated_2000_cycle_seconds_excluding_validation": 2000 * cycle_seconds}))
    rss_frame.plot(x="elapsed_seconds", y="rss_mib", title="Smoke-run process RSS (sampled)")
    plt.show()
    plt.close("all")
    print("Smoke checks passed. Review system memory pressure before the full fits.")
else:
    print("Smoke fit disabled.")

## Train the weight-30 experiment

In `"train"` mode, fit seed 0 for 2,000 updates. The weight-30 penalty encourages
return correlations at lags 1–5 to match training data; check whether volatility
and other statistics also remain realistic.

Save every validation checkpoint under `results/sde_gan/temporal_penalty_30/`
so we can compare training stages without refitting. Repeating this fit replaces
that run's files. The root model is still selected by lowest validation MMD.

In [ ]:
if MODE == "train":
    models, histories = {}, {}
    for seed in TRAINING_SEEDS:
        model = SDEGAN(SDEGANConfig(temporal_penalty_weight=TEMPORAL_PENALTY_WEIGHT,
                                    retain_validation_checkpoints=True))
        histories[seed] = pd.DataFrame(model.fit(train_windows, validation_windows, seed=seed))
        run_dir = results_dir / f"seed_{seed}"
        verify_checkpoint(model, run_dir)
        for directory in [run_dir, *sorted((run_dir / "checkpoints").glob("step_*"))]:
            (directory / "preprocessing.json").write_text(
                json.dumps(preprocessing_metadata, indent=2, allow_nan=False) + "\n"
            )
        models[seed] = model
    display(pd.DataFrame([{"seed": seed, **model.run_metadata}
                          for seed, model in models.items()]))
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for seed, history in histories.items():
        axes[0, 0].plot(history.step, history.generator_loss, label=f"seed {seed} generator")
        axes[0, 0].plot(history.step, history.critic_loss, alpha=0.6, label=f"seed {seed} critic")
        axes[0, 1].plot(history.step, history.gradient_penalty, label=f"seed {seed}")
        for metric in ["generator_grad_norm", "critic_grad_norm", "input_grad_norm"]:
            axes[1, 0].plot(history.step, history[metric], label=f"{seed}: {metric}")
        scores = history.dropna(subset=["validation_mmd2"])
        axes[1, 1].plot(scores.step, scores.validation_mmd2, marker=".", label=f"seed {seed}")
    for ax, title in zip(axes.flat, ["Losses", "Gradient penalty", "Gradient norms", "Validation MMD²"]):
        ax.set(title=title, xlabel="Generator update")
        ax.legend(fontsize=7)
    plt.show()
    plt.close(fig)
else:
    print("Full stock training disabled.")

if MODE == "train":
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    for seed, history in histories.items():
        axes[0].plot(history.step, history.generator_adversarial_loss, label=f"{seed}: adversarial")
        axes[0].plot(history.step, TEMPORAL_PENALTY_WEIGHT * history.temporal_penalty,
                     linestyle="--", label=f"{seed}: weighted temporal penalty")
        axes[1].plot(history.step, history.temporal_penalty, label=f"seed {seed}")
        scores = history.dropna(subset=["validation_temporal_error"])
        axes[2].plot(scores.step, scores.validation_temporal_error, marker=".", label=f"seed {seed}")
    for ax, title in zip(axes, ["Generator loss components", "Training temporal penalty (unweighted)",
                               "Validation log-return ACF error"]):
        ax.set(title=title, xlabel="Generator update")
        ax.legend(fontsize=7)
    plt.show()
    plt.close(fig)


## Compare saved checkpoints

Compare distribution mismatch (MMD), return-correlation error, and volatility
error. Lower is better; a volatility error of `0.2` means a 20% RMS relative
mismatch. Highlighted candidates show tradeoffs between the three metrics.
The weight-10 run has no logged volatility-error curve.

To inspect a saved step without retraining, use `MODE = "inspect"` and set
`INSPECT_CHECKPOINT_STEP`, for example to `800`. `None` uses the MMD-selected model.

In [ ]:
VALIDATION_METRICS = ["validation_mmd2", "validation_temporal_error", "validation_volatility_error"]


def pareto_candidates(table: pd.DataFrame) -> np.ndarray:
    """Identify validation rows not dominated across the three error metrics."""
    values = table[VALIDATION_METRICS].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Checkpoint comparison requires finite values for all three metrics")
    return np.array([not np.any(np.all(values <= row, axis=1) & np.any(values < row, axis=1))
                     for row in values])


fixture = pd.DataFrame([[1., 1., 1.], [2., 2., 2.], [.5, 2., 1.]], columns=VALIDATION_METRICS)
np.testing.assert_array_equal(pareto_candidates(fixture), [True, False, True])
if MODE in {"inspect", "train"}:
    for seed in TRAINING_SEEDS:
        run_dir = results_dir / f"seed_{seed}"
        index_path = run_dir / "validation_checkpoints.csv"
        if not index_path.is_file():
            print(f"Seed {seed}: no retained checkpoint index yet. Run the new full fit first.")
            continue
        checkpoint_table = pd.read_csv(index_path).sort_values("step")
        checkpoint_table["pareto_candidate"] = pareto_candidates(checkpoint_table)
        run_metadata = json.loads((run_dir / "run.json").read_text())
        checkpoint_table["mmd_selected"] = checkpoint_table.step.eq(run_metadata["selected_step"])
        chosen_step = (run_metadata["selected_step"] if INSPECT_CHECKPOINT_STEP is None
                       else INSPECT_CHECKPOINT_STEP)
        if chosen_step not in checkpoint_table.step.values:
            raise ValueError(f"Step {chosen_step} is not in this run's validation checkpoint index")
        checkpoint_table["inspected"] = checkpoint_table.step.eq(chosen_step)
        checkpoint_table["acf_rmse"] = np.sqrt(checkpoint_table.validation_temporal_error)
        checkpoint_table.to_csv(run_dir / "checkpoint_review.csv", index=False)
        print(f"Seed {seed}: all retained validation steps (lower errors are better)")
        display(checkpoint_table)
        print("Pareto candidates:")
        display(checkpoint_table.loc[checkpoint_table.pareto_candidate])
        fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
        for label, directory in [("weight 10", baseline_results_dir / "temporal_penalty"),
                                 ("weight 30", results_dir)]:
            history_path = directory / f"seed_{seed}" / "history.csv"
            if not history_path.is_file():
                continue
            previous = pd.read_csv(history_path)
            for metric, ax in zip(VALIDATION_METRICS, axes):
                if metric in previous:
                    values = previous.dropna(subset=[metric])
                    ax.plot(values.step, values[metric], marker=".", label=label)
        for metric, ax in zip(VALIDATION_METRICS, axes):
            candidates = checkpoint_table.loc[checkpoint_table.pareto_candidate]
            ax.scatter(candidates.step, candidates[metric], marker="*", s=85,
                       color="black", label="weight-30 Pareto candidates")
            ax.axvline(chosen_step, linestyle=":", color="grey", label="inspected step")
            ax.set(title=metric.replace("validation_", "").replace("_", " "), xlabel="Generator update")
            ax.legend(fontsize=7)
        fig.savefig(run_dir / "checkpoint_tradeoffs.png", dpi=150)
        plt.show()
        plt.close(fig)
        if INSPECT_CHECKPOINT_STEP is not None:
            if "models" not in globals():
                models = {}
            models[seed] = SDEGAN.load(inspection_run_dir(seed))
        print(f"Inspecting {inspection_run_dir(seed)}; no checkpoint is overwritten by this choice.")

## Calculate return and portfolio statistics

Calculate quantiles, correlations, and joint downside frequencies to compare
observed and generated returns. Downside thresholds come from training data so
all datasets use the same definition.

The portfolio rebalances to 25% per stock each session and compounds simple
returns. Its 95% loss VaR summarizes the upper tail of portfolio losses.

In [ ]:
def return_autocorrelation(paths: np.ndarray, absolute: bool = False) -> pd.DataFrame:
    """Correlate within-path pairs at lags 1–5, pooling windows without joining them."""
    values = np.abs(paths) if absolute else paths
    result = pd.DataFrame(index=range(1, 6), columns=STOCKS, dtype=float)
    for lag in result.index:
        left = pd.DataFrame(values[:, :-lag].reshape(-1, 4), columns=STOCKS)
        right = pd.DataFrame(values[:, lag:].reshape(-1, 4), columns=STOCKS)
        result.loc[lag] = left.corrwith(right)
    result.index.name = "lag"
    return result


def source_simple_windows(split: str) -> np.ndarray:
    """Read retained observed windows from CSV values, preserving exact zeros."""
    return np.stack([
        returns[split].loc[row.start_date:row.end_date].to_numpy()
        for row in window_dates[split].itertuples()
    ])


def original_simple(paths: np.ndarray, metadata: dict) -> np.ndarray:
    """Invert standardized samples using the scaler paired with their generator."""
    if metadata["stock_order"] != STOCKS:
        raise ValueError("Checkpoint stock order differs from this notebook")
    with np.errstate(over="raise", invalid="raise"):
        simple = np.expm1(paths.astype(np.float64) * np.asarray(metadata["training_std"])
                          + np.asarray(metadata["training_mean"]))
    if not np.isfinite(simple).all():
        raise ValueError("Nonfinite inverse-transformed returns")
    return simple


def terminal_returns(simple: np.ndarray) -> np.ndarray:
    """Compound all 20 daily returns of the rebalanced portfolio."""
    daily = simple @ WEIGHTS
    result = np.prod(1 + daily, axis=1) - 1
    if not np.isfinite(result).all():
        raise ValueError("Nonfinite terminal portfolio returns")
    return result


def path_statistics(simple: np.ndarray, thresholds: np.ndarray) -> dict[str, pd.DataFrame]:
    """Compute descriptive marginals, dependence, temporal, and portfolio summaries."""
    flat = pd.DataFrame(simple.reshape(-1, 4), columns=STOCKS)
    marginals = pd.DataFrame({"mean": flat.mean(), "sample_std": flat.std(ddof=1),
                              "zero_fraction": flat.eq(0).mean()})
    for q in [0.01, 0.05, 0.5, 0.95, 0.99]:
        marginals[f"q{q:.2f}"] = flat.quantile(q)
    acf = return_autocorrelation(simple, absolute=True)
    downside = flat.to_numpy() <= thresholds
    groups = [*combinations(range(4), 2), (0, 1, 2, 3)]
    tails = pd.DataFrame([
        {"stocks": " / ".join(STOCKS[i] for i in group),
         "count": int(downside[:, group].all(axis=1).sum()), "denominator": len(flat),
         "frequency": downside[:, group].all(axis=1).mean()}
        for group in groups
    ]).set_index("stocks")
    terminal = terminal_returns(simple)
    portfolio = pd.DataFrame([{
        "paths": len(simple), "mean_terminal_return": terminal.mean(),
        "terminal_q05": np.quantile(terminal, 0.05),
        "terminal_q50": np.quantile(terminal, 0.5),
        "terminal_q95": np.quantile(terminal, 0.95),
        "loss_var_95": np.quantile(-terminal, 0.95),
    }])
    return {"marginals": marginals, "pearson": flat.corr(),
            "spearman": flat.corr(method="spearman"), "absolute_acf": acf,
            "return_acf": return_autocorrelation(simple),
            "joint_downside": tails, "portfolio": portfolio}


# Hand-calculated portfolio fixture: a 10% move in one stock gives 2.5% portfolio return.
portfolio_fixture = np.zeros((1, 20, 4))
portfolio_fixture[0, 0, 0] = 0.10
np.testing.assert_allclose(terminal_returns(portfolio_fixture), [0.025])
portfolio_fixture[0, 1, 1] = -0.10
np.testing.assert_allclose(terminal_returns(portfolio_fixture), [1.025 * 0.975 - 1])

# Known correlations and boundary handling, independently of the model implementation.
acf_fixture = np.broadcast_to(np.array([-1., 1.])[:, None, None], (2, 20, 4))
np.testing.assert_allclose(return_autocorrelation(acf_fixture), 1)
# Alternating returns must have negative lag-1 correlation, not positive persistence.
alternating = np.broadcast_to(np.tile([1., -1.], 10)[None, :, None], (3, 20, 4))
np.testing.assert_allclose(return_autocorrelation(alternating).loc[1], -1)


## Check day-to-day dependence

Compare the original, weight-10, and weight-30 models with observed data.
Signed-return correlations reveal persistent directions; absolute-return
correlations show whether large moves cluster. Use pairs within each path,
never across path boundaries. Observed windows overlap, so comparisons are descriptive.

In [ ]:
if MODE in {"inspect", "train"}:
    temporal_datasets = {s: source_simple_windows(s) for s in ["train", "validation"]}
    for group, group_dir in [("baseline", baseline_results_dir),
                              ("weight 10", baseline_results_dir / "temporal_penalty"),
                              ("weight 30", results_dir)]:
        for seed in TRAINING_SEEDS:
            run_dir = inspection_run_dir(seed) if group == "weight 30" else group_dir / f"seed_{seed}"
            if not (run_dir / "generator.eqx").is_file():
                print(f"No saved {group} seed {seed} yet; skipping.")
                continue
            run_metadata = json.loads((run_dir / "preprocessing.json").read_text())
            if run_metadata != preprocessing_metadata:
                raise ValueError(f"{group} seed {seed} uses different preprocessing")
            restored = SDEGAN.load(run_dir)
            samples = restored.sample(n_paths=N_PATHS, seed=SAMPLE_SEED)
            temporal_datasets[f"{group} seed {seed}"] = original_simple(samples, run_metadata)
    diagnostics_dir = results_dir / "temporal_diagnostics"
    if INSPECT_CHECKPOINT_STEP is not None:
        diagnostics_dir = diagnostics_dir / f"step_{INSPECT_CHECKPOINT_STEP:04d}"
    diagnostics_dir.mkdir(parents=True, exist_ok=True)
    signed_acfs = {name: return_autocorrelation(a) for name, a in temporal_datasets.items()}
    log_acfs = {name: return_autocorrelation(np.log1p(a)) for name, a in temporal_datasets.items()}
    absolute_acfs = {name: return_autocorrelation(a, absolute=True) for name, a in temporal_datasets.items()}
    for label, tables in [("simple_return_acf", signed_acfs), ("log_return_acf", log_acfs),
                          ("absolute_return_acf", absolute_acfs)]:
        pd.concat(tables, names=["dataset"]).to_csv(diagnostics_dir / f"{label}.csv")
    print("Lag-1 correlation of signed simple returns (all sampled paths):")
    display(pd.DataFrame({name: table.loc[1] for name, table in signed_acfs.items()}).T.round(3))
    print("Lag-1 correlation of log returns (penalty units):")
    display(pd.DataFrame({name: table.loc[1] for name, table in log_acfs.items()}).T.round(3))
    print("Pooled daily simple-return volatility (%):")
    display(pd.DataFrame({name: a.reshape(-1, 4).std(axis=0, ddof=1) * 100
                          for name, a in temporal_datasets.items()}, index=STOCKS).T.round(3))
    for label, tables in [("signed_return", signed_acfs), ("absolute_return", absolute_acfs)]:
        fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, constrained_layout=True)
        for stock, ax in zip(STOCKS, axes.flat):
            for name, table in tables.items():
                if name in ["train", "validation"]:
                    style = {"color": "black" if name == "train" else "grey", "linewidth": 2.5}
                else:
                    seed = int(name.rsplit(" ", 1)[1])
                    style = {"color": f"C{seed}", "linestyle": "--" if name.startswith("baseline") else ":" if name.startswith("weight 10") else "-"}
                ax.plot(table.index, table[stock], marker=".", label=name, **style)
            ax.axhline(0, color="black", linewidth=0.5)
            ax.set(title=stock, xlabel="Lag (sessions)", ylabel="Correlation", xticks=range(1, 6))
            ax.legend(fontsize=7)
        fig.suptitle(f"{label.replace('_', ' ').capitalize()} autocorrelation: observed and generated")
        fig.savefig(diagnostics_dir / f"{label}_acf.png", dpi=150)
        plt.show()
        plt.close(fig)
    (diagnostics_dir / "settings.json").write_text(json.dumps({
        "sample_seed": SAMPLE_SEED, "n_paths": N_PATHS,
        "temporal_penalty_weight": TEMPORAL_PENALTY_WEIGHT, "inspected_step": INSPECT_CHECKPOINT_STEP, "lags": [1, 2, 3, 4, 5],
    }, indent=2) + "\n")
else:
    print("Temporal diagnostics run in inspect and train modes.")

## Compare generated and validation returns

In `"inspect"` and `"train"` modes, sample the chosen checkpoints and compare
return distributions and portfolio outcomes with observed data. This checks
whether the generated scenarios reproduce important market features.

Step 0.25 is the working default, supported by the summaries checked so far.
Use `inspect_sde_gan.ipynb` to recheck it after substantial model or horizon changes.

In [ ]:
if MODE in {"inspect", "train"}:
    observed_simple = {s: source_simple_windows(s)
                       for s in ["train", "validation"]}
    downside_thresholds = np.quantile(observed_simple["train"].reshape(-1, 4), 0.05, axis=0)
    generated_simple = {}
    validation_summaries = {}
    for seed in TRAINING_SEEDS:
        run_dir = inspection_run_dir(seed)
        model = SDEGAN.load(run_dir)
        paired_metadata = json.loads((run_dir / "preprocessing.json").read_text())
        generated = model.sample(n_paths=N_PATHS, seed=SAMPLE_SEED)
        assert generated.shape == (N_PATHS, 20, 4) and np.isfinite(generated).all()
        generated_simple[seed] = original_simple(generated, paired_metadata)
        np.savez_compressed(run_dir / "generated_paths.npz", standardized_log_returns=generated)
        datasets = {**observed_simple, "generated": generated_simple[seed]}
        validation_summaries[seed] = {name: path_statistics(a, downside_thresholds)
                                      for name, a in datasets.items()}
        for metric in validation_summaries[seed]["train"]:
            table = pd.concat({name: stats[metric] for name, stats in validation_summaries[seed].items()},
                              names=["dataset"])
            table.to_csv(run_dir / f"validation_{metric}.csv")
            print(f"Seed {seed}: {metric}")
            display(table)
        (run_dir / "evaluation.json").write_text(json.dumps({
            "sample_seed": SAMPLE_SEED, "n_paths": N_PATHS, "portfolio_weights": WEIGHTS.tolist(),
            "downside_training_thresholds": downside_thresholds.tolist(),
            "var_confidence": 0.95, "rebalancing": "each session",
        }, indent=2, allow_nan=False) + "\n")
    representative = {**observed_simple, "generated": generated_simple[0]}
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for ax, stock in zip(axes.flat, range(4)):
        for name, array in representative.items():
            values = np.sort(array[:, :, stock].ravel())
            ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=name)
        ax.set(title=STOCKS[stock], xlabel="Simple daily return", ylabel="Empirical CDF")
        ax.legend()
    plt.show()
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    for name, array in representative.items():
        values = np.sort(terminal_returns(array))
        ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=name)
    ax.set(title="20-session equal-weight rebalanced portfolio", xlabel="Terminal simple return",
           ylabel="Empirical CDF")
    ax.legend()
    plt.show()
    plt.close(fig)
else:
    print("Validation comparisons run in inspect and train modes.")

## Final test evaluation

Use `MODE = "test"` only after freezing model and evaluation choices. Load the
saved checkpoints and scaler to assess performance on untouched data without
refitting. Overlapping windows and few extreme losses limit the conclusions.

In [ ]:
if MODE == "test":
    os.environ["JAX_PLATFORMS"] = "cpu"
    if str(experiment_dir) not in sys.path:
        sys.path.insert(0, str(experiment_dir))
    from sde_gan import SDEGAN, SDEGANConfig

    for seed in TRAINING_SEEDS:
        run_dir = inspection_run_dir(seed)
        paired_metadata = json.loads((run_dir / "preprocessing.json").read_text())
        if paired_metadata != preprocessing_metadata:
            raise ValueError("Prepared data settings differ from this checkpoint's preprocessing")
        settings = json.loads((run_dir / "evaluation.json").read_text())
        if settings["portfolio_weights"] != WEIGHTS.tolist():
            raise ValueError("Notebook portfolio weights differ from frozen evaluation settings")
        restored = SDEGAN.load(run_dir)
        fake = original_simple(restored.sample(n_paths=settings["n_paths"],
                                               seed=settings["sample_seed"]), paired_metadata)
        real = source_simple_windows("test")
        summaries = {name: path_statistics(array, np.asarray(settings["downside_training_thresholds"]))
                     for name, array in {"test": real, "generated": fake}.items()}
        for metric in summaries["test"]:
            table = pd.concat({name: stats[metric] for name, stats in summaries.items()}, names=["dataset"])
            table.to_csv(run_dir / f"test_{metric}.csv")
            print(f"Seed {seed}: test {metric}")
            display(table)
else:
    print("Test evaluation runs only in test mode, after experiment choices are frozen.")